In [ ]:
# !pip install -q --upgrade langchain langchain-huggingface langchain-text-splitters langchain-community faiss-cpu sentence-transformers transformers pandas
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install sentence-transformers transformers langchain langchain-huggingface langchain-text-splitters langchain-community faiss-cpu pandas

In [10]:
import pandas as pd

In [11]:
df = pd.read_csv("helpdesk_202603231027.csv")


In [12]:

df.columns = df.columns.str.strip()

df.columns

Index(['id', 'actor_facing_issue', 'actor_facing_issue_date', 'resolving_date',
       'current_actor', 'caller_emp_id', 'helpdesk_member_name',
       'issue_description', 'solution_provided', 'category_id', 'created_on',
       'created_by', 'updated_on', 'updated_by', 'status', 'assigned_date',
       'allocated_by', 'actor_facing_type', 'subcategory_id', 'caller_role',
       'incident_id'],
      dtype='object')

In [13]:
df.head(2)

,id,actor_facing_issue,actor_facing_issue_date,resolving_date,current_actor,caller_emp_id,helpdesk_member_name,issue_description,solution_provided,category_id,...,created_by,updated_on,updated_by,status,assigned_date,allocated_by,actor_facing_type,subcategory_id,caller_role,incident_id
0,25965,r2444428,2026-03-21,2026-03-21,NaN,r2444428,Anurag Rai,Retailer called and said that while login the ...,So I told him that do clear data of my lava an...,4,...,Anurag Rai,NaN,NaN,resolved,NaN,NaN,Retailer,41,Retailer,NaN
1,25964,306931,2026-03-21,2026-03-21,NaN,306931,Anurag Rai,DBR called and said that while creating the Ds...,When I checked I found that the number matches...,4,...,Anurag Rai,NaN,NaN,resolved,NaN,NaN,Distributor,71,Distributor,NaN


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25965 entries, 0 to 25964
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       25965 non-null  int64  
 1   actor_facing_issue       25965 non-null  object 
 2   actor_facing_issue_date  25965 non-null  object 
 3   resolving_date           24993 non-null  object 
 4   current_actor            113 non-null    float64
 5   caller_emp_id            25965 non-null  object 
 6   helpdesk_member_name     25965 non-null  object 
 7   issue_description        25965 non-null  object 
 8   solution_provided        25122 non-null  object 
 9   category_id              25965 non-null  int64  
 10  created_on               25965 non-null  object 
 11  created_by               25965 non-null  object 
 12  updated_on               1312 non-null   object 
 13  updated_by               1312 non-null   object 
 14  status                

In [15]:
# convert to docyuments
from langchain_core.documents import Document

docs = []
for i, row in df.iterrows():
    content = f"Question: {row['issue_description']}\nAnswer: {row['solution_provided']}"
    docs.append(Document(page_content=content))

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 64}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
from langchain_community.vectorstores import FAISS
from tqdm import tqdm
import gc
import psutil
import os

process = psutil.Process(os.getpid())

batch_size = 500   # ideal for 16GB RAM
vectorstore = None

for i in tqdm(range(0, len(docs), batch_size)):

    batch = docs[i:i+batch_size]

    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        vectorstore.add_documents(batch)

    # Monitor RAM
    ram = process.memory_info().rss / 1024 / 1024

    print(
        f"Batch {i} done | RAM used: {ram:.2f} MB"
    )

    gc.collect()

vectorstore.save_local("faiss_index")

print("FAISS index created successfully")


  2%|▊                                           | 1/52 [00:02<02:10,  2.56s/it]

Batch 0 done | RAM used: 1289.04 MB


  4%|█▋                                          | 2/52 [00:05<02:14,  2.70s/it]

Batch 500 done | RAM used: 1215.23 MB


  6%|██▌                                         | 3/52 [00:09<02:54,  3.55s/it]

...

 98%|██████████████████████████████████████████▏| 51/52 [06:08<00:08,  8.21s/it]

Batch 25000 done | RAM used: 1302.43 MB


100%|███████████████████████████████████████████| 52/52 [06:15<00:00,  7.23s/it]

Batch 25500 done | RAM used: 1312.78 MB


FAISS index created successfully


In [18]:
query = input("your user question")

docs = vectorstore.similarity_search(query, k=3)

print(docs[0].page_content)

your user question DSE called and said he wants to know the retailer delete process


Question: DSE called and said he wants to know the retailer delete process
Answer: so i told him about it 


In [20]:
!pip install -q gradio

In [1]:


import gradio as gr


def answer_query(query):

    docs = vectorstore.similarity_search(query, k=3)

    if not docs:
        return "No relevant solution found."

    content = docs[0].page_content

    # Extract only the Answer part
    if "Answer:" in content:
        answer = content.split("Answer:")[1].strip()
    else:
        answer = content

    return answer


iface = gr.Interface(
    fn=answer_query,
    inputs=gr.Textbox(
        label="Need help? Just ask!",
        lines=10,        # increase height
        max_lines=20
    ),
    outputs=gr.Textbox(
        label="Happy to help! Here's your solution!",
        lines=10,        # increase height
        max_lines=20
    ),
    title="LAVA Assist",
    description="Hi there! 😊 How may I assist you today?",
     
)


iface.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [1]:
import psutil
import os

process = psutil.Process(os.getpid())

print("RAM used (MB):", process.memory_info().rss / 1024 / 1024)

print("System RAM usage:")
print(psutil.virtual_memory())

RAM used (MB): 161.60546875
System RAM usage:
svmem(total=16412217344, available=9109065728, percent=44.5, used=6191915008, free=5604802560, active=7494926336, inactive=1422286848, buffers=249913344, cached=4365586432, shared=763297792, slab=512356352)
